In [11]:
!pip install -q pytorch-tabnet==4.1.0
!pip install -q rtdl==0.0.13 --no-deps
!pip install -q optuna==3.5.0

from pytorch_tabnet.tab_model import TabNetClassifier
import rtdl
import optuna
print(f"✅ pytorch-tabnet: OK")
print(f"✅ rtdl: {rtdl.__version__}")
print(f"✅ optuna: {optuna.__version__}")

✅ pytorch-tabnet: OK
✅ rtdl: 0.0.13
✅ optuna: 3.5.0


In [12]:
import numpy as np
import pandas as pd
import os
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, roc_auc_score, f1_score)

os.makedirs('/kaggle/working/results', exist_ok=True)
os.makedirs('/kaggle/working/data', exist_ok=True)

SEEDS = [0, 42, 123]
print("✅ Libraries loaded")

✅ Libraries loaded


In [13]:
raw = fetch_california_housing(as_frame=True)
df_cal = raw.frame.copy()
print(f"California Housing shape: {df_cal.shape}")
print(df_cal.head(3))

California Housing shape: (20640, 9)
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  


In [14]:
url_train = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
url_test  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

col_names = [
    'age','workclass','fnlwgt','education','education_num',
    'marital_status','occupation','relationship','race','sex',
    'capital_gain','capital_loss','hours_per_week','native_country','income'
]

df_adult_tr = pd.read_csv(url_train, header=None, names=col_names,
                          skipinitialspace=True, na_values='?')
df_adult_te = pd.read_csv(url_test,  header=None, names=col_names,
                          skipinitialspace=True, na_values='?', skiprows=1)

df_adult = pd.concat([df_adult_tr, df_adult_te], ignore_index=True)

df_adult['income'] = df_adult['income'].str.replace('.','',regex=False).str.strip()
df_adult['income'] = (df_adult['income'] == '>50K').astype(int)

print(f"Adult Income shape: {df_adult.shape}")
print(df_adult['income'].value_counts())

Adult Income shape: (48842, 15)
income
0    37155
1    11687
Name: count, dtype: int64


In [15]:
from sklearn.datasets import fetch_covtype

raw_cov = fetch_covtype(as_frame=True)
df_cov = raw_cov.frame.copy()

df_cov = df_cov.sample(n=100_000, random_state=42).reset_index(drop=True)
df_cov['Cover_Type'] = df_cov['Cover_Type'] - 1 

print(f"Covertype shape: {df_cov.shape}")
print(df_cov['Cover_Type'].value_counts().sort_index())

Covertype shape: (100000, 55)
Cover_Type
0    36664
1    48661
2     6143
3      447
4     1696
5     2945
6     3444
Name: count, dtype: int64


In [16]:
def preprocess_dataset(df, target_col, task, seed=42):
    """
    输入 DataFrame，返回处理好的 X_train/val/test, y_train/val/test
    以及 feature_names, cat_indices（类别列的列号，DL模型需要）
    
    task: 'regression' or 'classification'
    """
    df = df.copy()
    
    # ── 1. 分离特征和标签 ─────────────────────────────
    X = df.drop(columns=[target_col])
    y = df[target_col].values
    
    # ── 2. 区分数值列和类别列 ──────────────────────────
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    
    # ── 3. 类别列：填缺失 + LabelEncode ───────────────
    for col in cat_cols:
        X[col] = X[col].fillna('__missing__')
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    
    # ── 4. 数值列：填缺失（中位数）────────────────────
    for col in num_cols:
        X[col] = X[col].fillna(X[col].median())
    
    # ── 5. 确保列顺序固定 ─────────────────────────────
    feature_cols = num_cols + cat_cols
    X = X[feature_cols].values.astype(np.float32)
    
    # ── 6. 60/20/20 split ─────────────────────────────
    X_tr, X_temp, y_tr, y_temp = train_test_split(
        X, y, test_size=0.40, random_state=seed,
        stratify=y if task=='classification' else None
    )
    X_val, X_te, y_val, y_te = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed,
        stratify=y_temp if task=='classification' else None
    )
    
    # ── 7. 数值列标准化（用训练集 fit）────────────────
    n_num = len(num_cols)
    scaler = StandardScaler()
    X_tr[:, :n_num]  = scaler.fit_transform(X_tr[:, :n_num])
    X_val[:, :n_num] = scaler.transform(X_val[:, :n_num])
    X_te[:, :n_num]  = scaler.transform(X_te[:, :n_num])
    
    # 类别列的列号（DL模型需要知道哪些是类别列）
    cat_indices = list(range(n_num, n_num + len(cat_cols)))
    
    print(f"  Train: {X_tr.shape}, Val: {X_val.shape}, Test: {X_te.shape}")
    print(f"  Numeric cols: {len(num_cols)}, Categorical cols: {len(cat_cols)}")
    
    return (X_tr, X_val, X_te,
            y_tr.astype(np.float32) if task=='regression' else y_tr.astype(np.int64),
            y_val.astype(np.float32) if task=='regression' else y_val.astype(np.int64),
            y_te.astype(np.float32)  if task=='regression' else y_te.astype(np.int64),
            feature_cols, cat_indices, scaler)

print("✅ preprocess_dataset() defined")

✅ preprocess_dataset() defined


In [17]:
print("=" * 50)
print("📦 Dataset 1: California Housing")
cal_data = preprocess_dataset(df_cal, target_col='MedHouseVal', task='regression')

print("=" * 50)
print("📦 Dataset 2: Adult Income")
adult_data = preprocess_dataset(df_adult, target_col='income', task='classification')

print("=" * 50)
print("📦 Dataset 3: Covertype")
cov_data = preprocess_dataset(df_cov, target_col='Cover_Type', task='classification')

# 保存处理好的数据，后续 notebook 直接加载
datasets = {
    'california': {'data': cal_data, 'task': 'regression',       'n_classes': 1},
    'adult':      {'data': adult_data,'task': 'classification',   'n_classes': 2},
    'covertype':  {'data': cov_data,  'task': 'multiclass',       'n_classes': 7},
}

with open('/kaggle/working/data/datasets.pkl', 'wb') as f:
    pickle.dump(datasets, f)

print("\n✅ All datasets saved to /kaggle/working/data/datasets.pkl")

📦 Dataset 1: California Housing
  Train: (12384, 8), Val: (4128, 8), Test: (4128, 8)
  Numeric cols: 8, Categorical cols: 0
📦 Dataset 2: Adult Income
  Train: (29305, 14), Val: (9768, 14), Test: (9769, 14)
  Numeric cols: 6, Categorical cols: 8
📦 Dataset 3: Covertype
  Train: (60000, 54), Val: (20000, 54), Test: (20000, 54)
  Numeric cols: 54, Categorical cols: 0

✅ All datasets saved to /kaggle/working/data/datasets.pkl


In [18]:
def evaluate(y_true, y_pred, y_prob, task):
    """
    返回一个 dict，包含所有需要汇报的指标
    y_prob: 预测概率（分类用），回归传 None
    """
    metrics = {}
    
    if task == 'regression':
        metrics['RMSE'] = np.sqrt(mean_squared_error(y_true, y_pred))
        metrics['MAE']  = mean_absolute_error(y_true, y_pred)
        metrics['R2']   = r2_score(y_true, y_pred)
        
    elif task == 'classification':
        metrics['Accuracy'] = accuracy_score(y_true, y_pred)
        metrics['AUC']      = roc_auc_score(y_true, y_prob[:, 1])
        metrics['F1']       = f1_score(y_true, y_pred, average='binary')
        
    elif task == 'multiclass':
        metrics['Accuracy'] = accuracy_score(y_true, y_pred)
        metrics['AUC']      = roc_auc_score(y_true, y_prob,
                                             multi_class='ovr', average='macro')
        metrics['F1']       = f1_score(y_true, y_pred, average='macro')
    
    return metrics

print("✅ evaluate() defined")

✅ evaluate() defined


In [20]:
import xgboost as xgb

def run_xgboost(dataset_name, datasets, seeds):
    d     = datasets[dataset_name]
    task  = d['task']
    X_tr, X_val, X_te, y_tr, y_val, y_te, feat_names, _, _ = d['data']
    
    all_metrics = []
    
    for seed in seeds:
        print(f"  seed={seed} ...", end=' ')
        
        # 参数（固定，不调参——baseline 不需要大动）
        if task == 'regression':
            params = dict(n_estimators=500, learning_rate=0.05,
                         max_depth=6, subsample=0.8, random_state=seed,
                         objective='reg:squarederror', device='cuda')
            model = xgb.XGBRegressor(**params)
        else:
            n_cls = d['n_classes']
            is_multi = n_cls > 2
            params = dict(
                n_estimators=500, learning_rate=0.05,
                max_depth=6, subsample=0.8, random_state=seed,
                eval_metric='mlogloss' if is_multi else 'logloss',
                objective='multi:softprob' if is_multi else 'binary:logistic',
                device='cuda'
            )
            if is_multi:
                params['num_class'] = n_cls
            model = xgb.XGBClassifier(**params)
        
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  verbose=False)
        
        # 预测
        if task == 'regression':
            y_pred = model.predict(X_te)
            y_prob = None
        else:
            y_prob = model.predict_proba(X_te)
            y_pred = np.argmax(y_prob, axis=1)
        
        m = evaluate(y_te, y_pred, y_prob, task)
        m['seed'] = seed
        all_metrics.append(m)
        print(f"done → {m}")
    
    # 汇总 mean ± std
    df_m = pd.DataFrame(all_metrics).drop(columns='seed')
    summary = {}
    for col in df_m.columns:
        summary[col] = f"{df_m[col].mean():.4f} ± {df_m[col].std():.4f}"
    
    return summary, model  # 返回最后一个 seed 的模型（用于特征重要性）

print("Running XGBoost on all datasets...")
xgb_results = {}

for ds_name in ['california', 'adult', 'covertype']:
    print(f"\n🔷 {ds_name.upper()}")
    summary, xgb_model = run_xgboost(ds_name, datasets, SEEDS)
    xgb_results[ds_name] = summary
    print(f"  Summary: {summary}")

# 保存
with open('/kaggle/working/results/xgboost_results.pkl', 'wb') as f:
    pickle.dump(xgb_results, f)

print("\n✅ XGBoost results saved")

Running XGBoost on all datasets...

🔷 CALIFORNIA
  seed=0 ... done → {'RMSE': np.float64(0.4586468855896101), 'MAE': 0.29938074946403503, 'R2': 0.8465808629989624, 'seed': 0}
  seed=42 ... done → {'RMSE': np.float64(0.460302643084474), 'MAE': 0.30156999826431274, 'R2': 0.8454711437225342, 'seed': 42}
  seed=123 ... done → {'RMSE': np.float64(0.45952788293688657), 'MAE': 0.30076906085014343, 'R2': 0.8459908962249756, 'seed': 123}
  Summary: {'RMSE': '0.4595 ± 0.0008', 'MAE': '0.3006 ± 0.0011', 'R2': '0.8460 ± 0.0006'}

🔷 ADULT
  seed=0 ... done → {'Accuracy': 0.8760364418057119, 'AUC': np.float64(0.9277619281305893), 'F1': 0.7164598454694451, 'seed': 0}
  seed=42 ... done → {'Accuracy': 0.8758317125601393, 'AUC': np.float64(0.9281996017193365), 'F1': 0.7154585972319962, 'seed': 42}
  seed=123 ... done → {'Accuracy': 0.8753198894462074, 'AUC': np.float64(0.9281993139276554), 'F1': 0.7147540983606557, 'seed': 123}
  Summary: {'Accuracy': '0.8757 ± 0.0004', 'AUC': '0.9281 ± 0.0003', 'F1': 

In [21]:
from sklearn.linear_model import LogisticRegression, Ridge

def run_linear(dataset_name, datasets, seeds):
    d    = datasets[dataset_name]
    task = d['task']
    X_tr, X_val, X_te, y_tr, y_val, y_te, _, _, _ = d['data']
    
    # 合并 train+val 来训练（linear model 不需要 early stopping）
    X_fit = np.concatenate([X_tr, X_val], axis=0)
    y_fit = np.concatenate([y_tr, y_val], axis=0)
    
    all_metrics = []
    
    for seed in seeds:
        print(f"  seed={seed} ...", end=' ')
        
        if task == 'regression':
            model = Ridge(alpha=1.0)
            model.fit(X_fit, y_fit)
            y_pred = model.predict(X_te)
            y_prob = None
        else:
            model = LogisticRegression(max_iter=1000, random_state=seed,
                                       n_jobs=-1, C=1.0)
            model.fit(X_fit, y_fit)
            y_prob = model.predict_proba(X_te)
            y_pred = np.argmax(y_prob, axis=1)
        
        m = evaluate(y_te, y_pred, y_prob, task)
        m['seed'] = seed
        all_metrics.append(m)
        print(f"done → {m}")
    
    df_m = pd.DataFrame(all_metrics).drop(columns='seed')
    summary = {}
    for col in df_m.columns:
        summary[col] = f"{df_m[col].mean():.4f} ± {df_m[col].std():.4f}"
    
    return summary

print("Running Linear models on all datasets...")
linear_results = {}

for ds_name in ['california', 'adult', 'covertype']:
    print(f"\n🔷 {ds_name.upper()}")
    summary = run_linear(ds_name, datasets, SEEDS)
    linear_results[ds_name] = summary
    print(f"  Summary: {summary}")

with open('/kaggle/working/results/linear_results.pkl', 'wb') as f:
    pickle.dump(linear_results, f)

print("\n✅ Linear results saved")

Running Linear models on all datasets...

🔷 CALIFORNIA
  seed=0 ... done → {'RMSE': np.float64(0.7425630121433998), 'MAE': 0.5341310501098633, 'R2': 0.5978488922119141, 'seed': 0}
  seed=42 ... done → {'RMSE': np.float64(0.7425630121433998), 'MAE': 0.5341310501098633, 'R2': 0.5978488922119141, 'seed': 42}
  seed=123 ... done → {'RMSE': np.float64(0.7425630121433998), 'MAE': 0.5341310501098633, 'R2': 0.5978488922119141, 'seed': 123}
  Summary: {'RMSE': '0.7426 ± 0.0000', 'MAE': '0.5341 ± 0.0000', 'R2': '0.5978 ± 0.0000'}

🔷 ADULT
  seed=0 ... done → {'Accuracy': 0.8249564950353158, 'AUC': np.float64(0.8497448266279599), 'F1': 0.546659597030753, 'seed': 0}
  seed=42 ... done → {'Accuracy': 0.8249564950353158, 'AUC': np.float64(0.8497448266279599), 'F1': 0.546659597030753, 'seed': 42}
  seed=123 ... done → {'Accuracy': 0.8249564950353158, 'AUC': np.float64(0.8497448266279599), 'F1': 0.546659597030753, 'seed': 123}
  Summary: {'Accuracy': '0.8250 ± 0.0000', 'AUC': '0.8497 ± 0.0000', 'F1': 

In [22]:
print("\n" + "="*60)
print("📊 BASELINE RESULTS SUMMARY")
print("="*60)

for ds_name in ['california', 'adult', 'covertype']:
    print(f"\n── {ds_name.upper()} ──")
    print(f"  XGBoost : {xgb_results[ds_name]}")
    print(f"  Linear  : {linear_results[ds_name]}")

print("\n✅ Day 1 完成！所有结果已保存到 /kaggle/working/results/")


📊 BASELINE RESULTS SUMMARY

── CALIFORNIA ──
  XGBoost : {'RMSE': '0.4595 ± 0.0008', 'MAE': '0.3006 ± 0.0011', 'R2': '0.8460 ± 0.0006'}
  Linear  : {'RMSE': '0.7426 ± 0.0000', 'MAE': '0.5341 ± 0.0000', 'R2': '0.5978 ± 0.0000'}

── ADULT ──
  XGBoost : {'Accuracy': '0.8757 ± 0.0004', 'AUC': '0.9281 ± 0.0003', 'F1': '0.7156 ± 0.0009'}
  Linear  : {'Accuracy': '0.8250 ± 0.0000', 'AUC': '0.8497 ± 0.0000', 'F1': '0.5467 ± 0.0000'}

── COVERTYPE ──
  XGBoost : {'Accuracy': '0.8481 ± 0.0002', 'AUC': '0.9819 ± 0.0000', 'F1': '0.8150 ± 0.0024'}
  Linear  : {'Accuracy': '0.7156 ± 0.0000', 'AUC': '0.9329 ± 0.0000', 'F1': '0.5071 ± 0.0000'}

✅ Day 1 完成！所有结果已保存到 /kaggle/working/results/
